# Platform Architecture

Suppose you have three compute providers: a RunPod GPU cluster, a pair of AWS EC2 instances, and a bare-metal server you SSH into with a key file. Each has a completely different API. RunPod has a REST endpoint, AWS has `boto3`, and the bare server has nothing but `openssh`. You want to submit a notebook training run and not care which provider runs it.

This is the problem **NBX** (Notebook Execute) solves. NBX is a lightweight compute platform that exposes a single, uniform interface over heterogeneous compute backends. You define a job, submit it, and NBX handles provider selection, packaging, execution, log streaming, and artifact retrieval. This notebook covers the design principles and data models that make that abstraction possible.

## System Design Principles

Before writing any code, it helps to state the principles that constrain the design. These are not aspirational — they are load-bearing. Violating any one of them causes a specific class of problems downstream.

**Single responsibility.** The platform routes work; it does not do work. NBX never trains a model, never executes a notebook cell itself. It packages, transfers, supervises, and reports. This boundary is strict: the moment the platform starts understanding the *content* of jobs, complexity explodes.

**Provider abstraction.** Every backend — RunPod, AWS, bare SSH — implements the same `BaseProvider` interface. Callers never speak directly to a provider. This makes providers swappable and testable in isolation with mocks.

**Job as the unit of work.** Everything is normalized to a `JobSpec`. Whether the job is a Python script, a Jupyter notebook run via papermill, or a multi-file project — it becomes a `JobSpec` before it enters the queue. The queue, the dispatcher, and the workers only speak `JobSpec`.

**Async-first.** All I/O — HTTP calls to provider APIs, SSH connections, file transfers, WebSocket log streams — is non-blocking. We use `asyncio` throughout. A blocking call anywhere in the hot path stalls the event loop and freezes the entire platform.

**12-factor alignment.** Three factors from the [12-factor app](https://12factor.net/) methodology are directly relevant here: (1) **config via environment** — no secrets in code, all credentials come from env vars via `pydantic-settings`; (2) **stateless processes** — the API server holds no job state in memory between requests, state lives in the database and queue; (3) **logs as streams** — every log line is an event emitted to stdout or a WebSocket stream, never written to a local file that dies with the process.

:::{.callout-note}
The 12-factor methodology was written for web apps deployed on Heroku-style platforms, but the three principles above apply equally to compute orchestration systems. The key insight is the same: decouple the application from the infrastructure it runs on.

:::

## The JobSpec Schema

The `JobSpec` is the lingua franca of the platform. Every subsystem — the API layer, the queue, the dispatcher, the runner — communicates via `JobSpec` objects. Getting this schema right is the most important design decision in the system.

A job has a **type** that determines how it is executed, an **entrypoint** that identifies the file to run, optional **params** passed at runtime, optional **env** for environment variable injection, and a **provider_id** hint for routing. We define a `JobType` enum to constrain the type field:

In [ ]:
from __future__ import annotations

import uuid
from datetime import datetime, timezone
from enum import Enum
from typing import Any

from pydantic import BaseModel, Field


class JobType(str, Enum):
    notebook = "notebook"
    script   = "script"
    project  = "project"

With `JobType` defined, we write the full `JobSpec` model:

In [ ]:
class JobSpec(BaseModel):
    id:          str      = Field(default_factory=lambda: uuid.uuid4().hex)
    type:        JobType
    entrypoint:  str                        # relative path within job root
    path:        str      = "."             # local project root to package
    params:      dict[str, Any] = {}
    env:         dict[str, str] = {}
    provider_id: str | None     = None      # None = let dispatcher choose
    created_at:  datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc)
    )

Validating a raw dict against the schema:

In [ ]:
raw = {
    "type": "notebook",
    "entrypoint": "train.ipynb",
    "params": {"epochs": 20, "lr": 3e-4},
    "env": {"CUDA_VISIBLE_DEVICES": "0"},
    "provider_id": "ssh-gpu-01",
}

spec = JobSpec.model_validate(raw)
print(spec.id)
print(spec.type)
print(spec.created_at)

The serialized form — what actually travels over the wire to the queue:

In [ ]:
print(spec.model_dump_json(indent=2))

**Remark.** We deliberately keep `JobSpec` flat — no nested objects beyond the `params` and `env` dicts. This makes serialization trivial and keeps the queue implementation simple. Nested specs tend to grow unboundedly as teams add fields; the discipline of keeping it flat forces clarity about what the scheduler actually needs to know.

## Provider Abstraction

The provider layer is where the platform touches actual infrastructure. Every provider — SSH host, RunPod, AWS EC2 — exposes the same four operations: list machines, get metrics, execute a job, and fetch artifacts. An **abstract base class** captures this contract, making the rest of the platform independent of any specific backend.

**The adapter pattern.** `BaseProvider` is a classic [adapter](https://en.wikipedia.org/wiki/Adapter_pattern): it wraps a heterogeneous API — RunPod REST, `asyncssh`, `boto3` — behind a uniform interface. The wrapper does two things: translates method calls and converts return types. `RunPodProvider.list_machines()` calls the RunPod REST API; `SSHProvider.list_machines()` reads a config file. Callers see only `BaseProvider.list_machines()`. This means every dispatcher, health poller, and unit test is written once and works against any provider.

**Why four operations.** The choice of four abstract methods is deliberately minimal — monitoring (`list_machines`, `get_metrics`) and task lifecycle (`execute`, `fetch_artifacts`). Authentication, scheduling, and retries are concerns of the concrete provider or the caller, not the abstract interface. Adding a fifth method to the ABC would break every existing concrete provider; the bar for adding one is high.

In [ ]:
import abc
import pathlib
from dataclasses import dataclass, field


class MachineStatus(str, Enum):
    online  = "online"
    offline = "offline"
    unknown = "unknown"


@dataclass
class MachineInfo:
    id:       str
    name:     str
    provider: str
    host:     str
    status:   MachineStatus = MachineStatus.unknown
    tags:     list[str]     = field(default_factory=list)

With the data types in place, we define `BaseProvider`:

In [ ]:
class BaseProvider(abc.ABC):
    """Uniform interface over a compute backend."""

    @abc.abstractmethod
    async def list_machines(self) -> list[MachineInfo]:
        """Return all machines known to this provider."""

    @abc.abstractmethod
    async def get_metrics(self, machine_id: str) -> dict[str, float]:
        """Return a metrics snapshot for a single machine."""

    @abc.abstractmethod
    async def execute(self, machine_id: str, job_spec: JobSpec) -> str:
        """Submit a job to a machine; return the remote job ID."""

    @abc.abstractmethod
    async def fetch_artifacts(
        self, job_id: str, dest: pathlib.Path
    ) -> list[pathlib.Path]:
        """Pull job artifacts to a local destination; return copied paths."""

Concrete providers subclass `BaseProvider` and implement the four methods for their specific API. The rest of the platform always holds a `BaseProvider` reference — never the concrete type:

In [ ]:
class SSHProvider(BaseProvider):
    """Executes jobs on machines reachable via SSH."""
    async def list_machines(self):              return NotImplemented
    async def get_metrics(self, mid):           return NotImplemented
    async def execute(self, mid, spec):         return NotImplemented
    async def fetch_artifacts(self, jid, dst):  return NotImplemented


class RunPodProvider(BaseProvider):
    """Executes jobs on RunPod GPU pods via the RunPod REST API."""
    async def list_machines(self):              return NotImplemented
    async def get_metrics(self, mid):           return NotImplemented
    async def execute(self, mid, spec):         return NotImplemented
    async def fetch_artifacts(self, jid, dst):  return NotImplemented


class AWSProvider(BaseProvider):
    """Executes jobs on EC2 instances via boto3 + SSM."""
    async def list_machines(self):              return NotImplemented
    async def get_metrics(self, mid):           return NotImplemented
    async def execute(self, mid, spec):         return NotImplemented
    async def fetch_artifacts(self, jid, dst):  return NotImplemented

:::{.callout-caution}
Any code that imports a concrete provider class by name has violated the abstraction. Pass providers around as `BaseProvider` references and let the startup configuration resolve the concrete type. This is the only way to keep the dispatcher and worker code testable with mocks.

:::

## Config & Secrets Management

Credentials — SSH key paths, API keys, database URLs — must never appear in source code. The 12-factor methodology is unambiguous on this: configuration that varies between deployments lives in environment variables. We use `pydantic-settings` to read those variables into a typed, validated settings object at startup.

`NbxSettings` covers four configuration domains: the API server, the database, SSH credentials, and cloud provider credentials:

In [ ]:
from pydantic_settings import BaseSettings, SettingsConfigDict


class NbxSettings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )

    # API server
    nbx_host: str = "0.0.0.0"
    nbx_port: int = 8000

    # Storage
    nbx_db_url: str = "sqlite+aiosqlite:///./nbx.db"

    # SSH provider
    ssh_key_path: str = "~/.ssh/id_rsa"

    # RunPod
    runpod_api_key: str = ""

    # AWS
    aws_access_key_id:     str = ""
    aws_secret_access_key: str = ""

`NbxSettings()` reads from environment variables (and, as a fallback, a `.env` file) at instantiation:

In [ ]:
cfg = NbxSettings()
print(f"API:          {cfg.nbx_host}:{cfg.nbx_port}")
print(f"DB:           {cfg.nbx_db_url}")
print(f"SSH key:      {cfg.ssh_key_path}")
print(f"RunPod set:   {bool(cfg.runpod_api_key)}")
print(f"AWS set:      {bool(cfg.aws_access_key_id)}")

A minimal `.env` file for a single-machine SSH deployment:

```{.bash filename=".env"}
NBX_HOST=0.0.0.0
NBX_PORT=8000
NBX_DB_URL=sqlite+aiosqlite:///./nbx.db
SSH_KEY_PATH=/home/user/.ssh/nbx_key
RUNPOD_API_KEY=
AWS_ACCESS_KEY_ID=
AWS_SECRET_ACCESS_KEY=
```

**NOTE:** Add `.env` to `.gitignore` immediately. The `NbxSettings` defaults allow the platform to start with no `.env` at all, which is useful in CI environments where secrets are injected as environment variables directly.

:::{.callout-important}
Never commit `.env` files. If a secret leaks into git history, rotate it immediately — `git` history edits are insufficient because the secret may already have been cloned or cached by CI systems.

:::

## Project Scaffold

The project is organized into focused modules — no module should need to import from more than one other module's domain. The full directory tree:

```
nbx/
  main.py              # FastAPI app factory: creates the app, mounts routers
  config.py            # NbxSettings singleton loaded at startup
  models.py            # JobSpec, MachineInfo, and all shared Pydantic models
  providers/
    base.py            # BaseProvider abstract class
    ssh.py             # SSHProvider: asyncssh-based execution
    runpod.py          # RunPodProvider: RunPod REST API wrapper
    aws.py             # AWSProvider: boto3 + SSM execution
  routers/
    machines.py        # /machines CRUD: register, list, update, delete
    jobs.py            # /jobs endpoints: submit, list, get, cancel
  queue/
    worker.py          # QueueWorker: pulls jobs, calls dispatcher, updates status
    dispatcher.py      # Dispatcher: provider selection and job routing
  streaming/
    websocket.py       # ConnectionManager: WebSocket log fan-out to N clients
    artifacts.py       # fetch_artifacts: SCP pull + FileResponse serving
  dashboard/
    main.py            # Flet app (standalone script — not part of FastAPI)
```

The dependency graph is strictly layered: `routers` → `queue` → `providers` → (external). The `dashboard` module stands entirely alone — it is a Flet application that calls the FastAPI service over HTTP/WebSocket and is never imported by the server.

The FastAPI app factory wires the pieces together:

In [ ]:
from fastapi import FastAPI


def create_app() -> FastAPI:
    app = FastAPI(title="NBX", version="0.1.0")

    # Routers registered here in the real implementation:
    # app.include_router(machines_router, prefix="/machines")
    # app.include_router(jobs_router,     prefix="/jobs")

    @app.get("/health")
    async def health():
        return {"status": "ok"}

    return app


app = create_app()

**Remark.** The `create_app()` factory pattern — rather than a module-level `app = FastAPI()` — makes the application testable. Tests can call `create_app()` with a test database URL and different config without any global state leaking between test runs.

With the architecture sketched, the data models defined, and the project structure established, the subsequent notebooks build each subsystem in depth. The next notebook starts with the machine registry — the address book that every other subsystem queries.

:::{.callout-note}
Throughout this series, code cells define the authoritative implementations. In a real project these would be proper Python packages in the `nbx/` directory shown above.

:::

---

■